# 05 — Regional Sales Analysis
Sales, profit, and order volume broken down by region and state.


In [ ]:
from pyhive import hive
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

# ── Global style ──────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = sns.color_palette("muted")

def get_conn():
    return hive.connect(host="hive-server2", port=10000,
                        database="default", auth="NONE")

def fetch_df(cur, sql):
    cur.execute(sql)
    cols = [d[0].split(".")[-1] for d in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

def fmt_usd(v):
    if abs(v) >= 1_000_000:
        return f"${v/1_000_000:.2f}M"
    if abs(v) >= 1_000:
        return f"${v/1_000:.1f}K"
    return f"${v:.0f}"

conn = get_conn()
cur  = conn.cursor()
print("Connected.")


In [ ]:
# ── Sales by Region ───────────────────────────────────────────────────────────
region = fetch_df(cur, """
    SELECT l.region,
           ROUND(SUM(oi.sales),2)  AS revenue,
           ROUND(SUM(oi.profit),2) AS profit,
           COUNT(DISTINCT o.order_id) AS orders,
           ROUND(SUM(oi.profit)/SUM(oi.sales)*100,2) AS margin_pct
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN locations l ON o.postal_code = l.postal_code
    GROUP BY l.region ORDER BY revenue DESC
""")

bar_w = 0.35
x     = range(len(region))
fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

b1 = ax1.bar([i - bar_w/2 for i in x], region["revenue"],
             bar_w, color=PALETTE[0], label="Revenue", zorder=3)
b2 = ax1.bar([i + bar_w/2 for i in x], region["profit"],
             bar_w, color=PALETTE[1], label="Profit", zorder=3)
ax2.plot(x, region["margin_pct"], color=PALETTE[2], marker="o",
         linewidth=2, markersize=7, label="Margin %", zorder=4)

for bar in b1:
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 1000,
             fmt_usd(bar.get_height()),
             ha="center", va="bottom", fontsize=8, color=PALETTE[0])
for bar in b2:
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 200,
             fmt_usd(bar.get_height()),
             ha="center", va="bottom", fontsize=8, color=PALETTE[1])
for xi, mi in zip(x, region["margin_pct"]):
    ax2.text(xi, mi + 0.5, f"{mi:.1f}%",
             ha="center", va="bottom", fontsize=8,
             color=PALETTE[2], fontweight="bold")

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2,
           loc="upper right", frameon=False, fontsize=9)
ax1.set_xticks(list(x))
ax1.set_xticklabels(region["region"], fontsize=11)
ax1.set_title("Revenue, Profit & Margin % by Region")
ax1.set_ylabel("USD")
ax2.set_ylabel("Margin %")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt_usd(v)))
ax1.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
plt.tight_layout()
plt.show()


In [ ]:
# ── Top 10 States by Revenue ─────────────────────────────────────────────────
states = fetch_df(cur, """
    SELECT l.state,
           ROUND(SUM(oi.sales),2)  AS revenue,
           ROUND(SUM(oi.profit),2) AS profit
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN locations l ON o.postal_code = l.postal_code
    GROUP BY l.state ORDER BY revenue DESC LIMIT 10
""")

fig, ax = plt.subplots(figsize=(10, 5))
y = range(len(states))
ax.barh([i + 0.2 for i in y], states["revenue"][::-1],
        0.4, color=PALETTE[0], label="Revenue")
ax.barh([i - 0.2 for i in y], states["profit"][::-1],
        0.4, color=PALETTE[1], label="Profit", alpha=0.85)
ax.set_yticks(list(y))
ax.set_yticklabels(states["state"][::-1])

for i, (rev, prof) in enumerate(zip(states["revenue"][::-1],
                                    states["profit"][::-1])):
    ax.text(rev + 200, i + 0.2, fmt_usd(rev), va="center", fontsize=8)
    ax.text(prof + 200, i - 0.2, fmt_usd(prof), va="center",
            fontsize=8, color=PALETTE[1])

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt_usd(v)))
ax.legend(frameon=False)
ax.set_xlabel("USD")
ax.set_title("Top 10 States: Revenue vs Profit")
ax.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
cur.close()
conn.close()
print("Done.")
